In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import joblib
import xgboost
import sklearn
import hashlib
from scipy import stats
from sklearn.pipeline import Pipeline

from src.estimators import NaivePersistence, DirectARBaseline, XGBoostDst
from src.evaluate import diebold_mariano
from src.runner import fit_model, run_segment
from src.mlflow_tracking import setup_mlflow
from src.config import STORM_THR, K_HORIZONS
from src.splits import build_masks
from src.features import build_feature_sets, NEUTRON_DERIVED

setup_mlflow()
# Reproducibility: single-threaded fits for the Validation stage (see note below —
# XGBRegressor's multi-threaded histogram construction is not bit-reproducible across
# runs even with a fixed random_state, due to non-associative floating-point summation
# across threads).
N_JOBS_VALIDATION = 1


# Geomagnetic Storm Prediction from Cosmic Ray Measurements — Validation of Neutron Flux Predictive Gain

See the Abstract and Introduction in `cosmic_ray_storm_prediction_EDA.ipynb` for the project's domain, data sources, and overall research objective.

## Introduction

This notebook isolates the **Validation of Neutron Flux Predictive Gain** stage from the main machine-learning workflow (`cosmic_ray_storm_prediction_ML.ipynb`), so that the choice of predictor set (MODEL_A–D) and forecasting horizon ($h^*$) can be examined and documented on its own, before any operational model tuning takes place.

It depends on artifacts produced by two earlier stages:

- **Feature Engineering** (`cosmic_ray_storm_prediction_FE.ipynb`) — constructs `data/processed/feat_split.parquet` and `models/context_constants.pkl`.
- **Feature Screening** (`feature_selection.ipynb`) — produces `models/feature_selection_results.pkl`, containing `selected_final` (the 24-predictor modelling subset) and the fitted `StandardScaler`.

**Research question:** do neutron-monitor-derived predictors provide predictive information for $D_{st}(t+h)$ beyond the conventional OMNI solar wind parameters, and at which forecasting horizon is that contribution best supported by the evidence?

| Model | Predictors | Role |
|---|---|---|
| MODEL_A | OMNI only | Baseline |
| MODEL_B | OMNI + `neutron_counts` (raw) | Raw neutron signal |
| MODEL_C | OMNI + $\delta n(t)$ | Differential neutron flux |
| MODEL_D | OMNI + $\delta n(t)$ + neutron lags | Neutron flux with history |

**The chronological split.** Every model in this project is trained and evaluated on the same six segments, carved out of the 1995–2023 record (see the Train/Validation/Test Split section of `cosmic_ray_storm_prediction_FE.ipynb` for the full rationale):

| Segment | Period | Role |
|---|---|---|
| `Train₁` | 1995-01-01 → 2003-10-14 | Fitting (Solar Cycle 22–23 rise) |
| `Val_Storm` | 2003-10-15 → 2003-12-15 | Carved out of the training period, reserved for storm diagnostics — the Halloween 2003 superstorm |
| `Train₂` | 2003-12-16 → 2008-12-31 | Fitting, continued (Solar Cycle 23 decline) |
| `Val_Main` | 2009-01-01 → 2014-12-31 | Routine validation (Solar Cycle 24) |
| `Test_Active` | 2015 | Held-out test — active period |
| `Test_Quiet` | 2016–2023 | Held-out test — quiet period |

Segment boundaries have a 21-hour purge zone on each side (the longest forecast horizon considered) to stop information from one segment leaking into the next through lagged features. `Val_Storm` is deliberately carved out of the training window rather than appended after it — following Nair et al. (2023) `[NAI23]` — so the training data still spans a full solar cycle while an extreme storm stays available for dedicated evaluation. **This notebook trains only on `Train₁`** (not `Train₂` — that's added later, in the Operational Forecasting Model stage) and evaluates on `Val_Main` and `Val_Storm`.


## Feature Screening and Selection

This stage supports the **forecasting task** defined in the Problem Formulation by identifying the predictor subset used consistently throughout all subsequent modelling experiments. Using a common feature set ensures that model comparisons reflect differences in forecasting methodology rather than differences in the available predictors.

The engineered feature matrix contains 33 candidate predictors. Before model development, a dedicated screening stage removes redundant variables while retaining those that consistently contribute to forecasting performance.

To keep the modelling workflow reproducible, feature screening is implemented in the helper notebook `feature_selection.ipynb`. The resulting predictor subset is then reused unchanged during both the Scientific Validation experiments and the Operational Forecasting Model.

Two complementary selection methods are combined: LASSO, which performs embedded linear feature selection, and ExtraTrees, which estimates non-linear feature importance. The final predictor subset is obtained by combining the selections across methods and forecasting horizons.

Using the strict selection criterion (`min_votes = 2`), 23 predictors are retained by both methods. Together with the physically motivated inclusion of `d_neutron`, this produces the final set of 24 predictors used throughout the remainder of the study.

### Feature Scaling

The predictors span different physical units and numerical ranges, including nanoteslas, kilometres per second, kelvin, and neutron counts. Before feature screening, the predictor matrix is standardised using `StandardScaler` so that all variables are expressed on a comparable scale.

Each feature is transformed to zero mean and unit variance:

$$
\tilde{x}_{t,j}=\frac{x_{t,j}-\mu_j}{\sigma_j},
$$

where $\mu_j$ and $\sigma_j$ are the mean and standard deviation of feature $j$ estimated from the training segment only. The fitted transformation is then applied unchanged to the validation and test segments, preventing distribution leakage.

Standardisation is performed only for the feature screening stage. The scaler is fitted using the training data and reused for all subsequent segments, ensuring that no information from future observations influences feature selection.

### Feature Screening

Feature screening is implemented in the helper notebook `feature_selection.ipynb`. Separating this stage from the main workflow avoids repeating the computationally intensive selection procedure while ensuring that all subsequent experiments use the same predictor subset.

The helper notebook loads the feature matrix and shared experimental configuration produced during Feature Engineering:

- `feat_split.parquet` — imputed, unscaled feature matrix;
- `split_masks.pkl` — train, validation, and test segment masks;
- `context_constants.pkl` — shared project constants (`FEATURE_COLS`, `K_HORIZONS`, `STORM_THR`, `PURGE_H`, and `BOUNDARIES`).

### Methodology

Feature selection combines two complementary methods implemented in the custom `FeatureSelectionPipeline`. The methods were chosen for three reasons: they capture both linear and non-linear predictor relationships, support sample weighting, and are compatible with the chronological training procedure adopted in this study.

- **LASSO** performs embedded linear feature selection using `LassoCV` with `TimeSeriesSplit (n_splits = 5)`. The $\ell_1$ penalty shrinks irrelevant coefficients exactly to zero, producing a compact predictor set while selecting the regularisation strength by cross-validation.

- **ExtraTrees** estimates feature importance using an ensemble of 100 extremely randomised trees. Unlike LASSO, it captures non-linear relationships and interactions between predictors without assuming linear dependence on $D_{st}$.

Both methods support `sample_weight`, allowing storm observations ($D_{st} < -50$ nT) to receive higher weight during feature selection. This is essential because storm periods are strongly underrepresented in the chronological training set. A predictor is retained only if it is selected by both methods for at least one forecast horizon (`min_votes = 2`).

Using the strict voting criterion (`min_votes = 2`), 23 predictors were selected consistently by both screening methods. The final modelling subset contains 24 predictors, with `d_neutron` retained as a physics-guided feature despite receiving a single vote. This preserves the derived neutron precursor signal while maintaining a predominantly data-driven selection procedure.

### Results

> **Feature Screening Results:**
> - The strict voting criterion (`min_votes = 2`) retained 23 predictors; the liberal criterion (`min_votes = 1`) retained all 33 candidate predictors — every feature is selected by at least one method at at least one horizon, and no feature has zero votes in the pipeline output. The final modelling subset contains 24 predictors, with `d_neutron` retained as a physics-guided feature despite receiving a single vote.
>
> - **LASSO** selected between 14 and 22 predictors depending on the forecast horizon (α ranges from 0.226 at h=12h to 1.293 at h=3h — the highest regularisation strength among the five horizons, not monotonic with h).
>
> - **ExtraTrees** retained exactly 17 predictors at every forecast horizon, driven by the fixed 50th-percentile importance threshold.
>
> - **`d_neutron`** survives storm-weighted LASSO at h=7h (coefficient 0.141838) but falls below the ExtraTrees median importance threshold at all five horizons — reflecting the event-driven nature of the Forbush Decrease signal, concentrated in a small fraction of records [KIS25].
>
> - `dbz_dt` has `Max votes = 1` — the same status as `d_neutron` and 8 other features excluded under the strict criterion. No feature is eliminated by both methods at all horizons.

### Physical interpretation of excluded features

Features excluded from the final modelling subset fall into two categories. The first consists of `dbz_dt`, which was rejected by both selection methods at every forecast horizon (`votes = 0`). The second comprises nine predictors receiving a single vote, indicating that they were supported by only one selection method and therefore lacked cross-method agreement.

`bz_gsm_lag7` is the southward IMF component measured 7 hours before the current observation — precisely at the Burton ring current decay timescale (τ ≈ 7–8 hours). The ring current acts as an integrator: the magnetic field state 7 hours ago is already encoded in the present $D_{st}$ value through exponential decay. Short lags (`bz_gsm_lag1`, `bz_gsm_lag3`) capture the onset of southward turning; the accumulated Bz features capture the injection history over 3, 6, and 12 hours. The absence of cross-method agreement suggests that its predictive information is already represented by the neighbouring lag and accumulated IMF features.

`sw_speed_lag21` is the solar wind bulk velocity measured 21 hours before the current observation. Long-term baseline solar wind conditions are better represented by `ssn` and `f107` already in the feature set, while the active CME driver is captured by shorter lags. No additional predictive information is expected beyond that already captured by the shorter lags and the long-term solar activity indicators.

`neutron_counts_lag1` is the LMKS neutron monitor count rate 1 hour before the current observation. The Forbush Decrease unfolds over 6–24 hours — a 1-hour lag is physically indistinguishable from the current value at hourly resolution. The signal is already captured by `neutron_counts` and more cleanly by `d_neutron`.

`neutron_counts_lag21` is the neutron monitor count rate at the outer boundary of the 7–21h Forbush Decrease lead time window [KIS25]. At this lag the precursor signal is already attenuating and does not add detectable information beyond the shorter retained lags.

`sw_temp` and `plasma_beta` Both variables were retained by LASSO at some horizons but consistently fell below the ExtraTrees importance threshold. Their contribution appears to overlap with the retained solar wind state variables (`sw_speed`, `sw_density`, and `sw_pressure`).

`dbz_dt` represents the hour-to-hour change in the southward IMF component. At hourly resolution it provides no consistent predictive contribution beyond the retained accumulated IMF features (`bz_acc_3h`, `bz_acc_6h`, `bz_acc_12h`), which better represent the sustained southward IMF driving ring current injection in the Burton model [BUR75]. Accordingly, `dbz_dt` was rejected by both selection methods at every forecast horizon.

## Validation of Neutron Flux Predictive Gain

This stage addresses the **research objective** and **forecasting horizons** defined in the Problem Formulation. Its purpose is to determine whether neutron-monitor observations provide predictive information beyond that already contained in the conventional OMNI solar wind measurements and to identify the most suitable operational forecasting horizon.

Rather than directly constructing the final forecasting model, the analysis proceeds through a sequence of controlled experiments. The temporal split, learning algorithm, training procedure, and evaluation protocol remain fixed throughout, while the available predictor set is progressively expanded. This controlled design ensures that any observed change in predictive performance can be attributed to the additional predictors rather than to differences in model configuration.

The validation consists of two consecutive stages:

1. **Establishing Predictability.** Quantify the predictive skill obtainable from persistence, the current geomagnetic state, and conventional OMNI solar wind observations, establishing the reference baselines for all subsequent comparisons.

2. **Neutron Flux Predictive Gain.** Progressively augment the OMNI baseline with neutron-derived predictors, quantify the incremental predictive value of each neutron representation, and identify the forecasting horizon at which neutron-derived information provides the greatest relative contribution.

The conclusions of this section determine whether neutron observations provide measurable predictive value and identify the forecast horizon and neutron representation carried forward to the Operational Forecasting Model.

**Evaluation metrics.** Model performance is assessed using four complementary metrics. **RMSE** measures the overall prediction error, while **Storm RMSE** evaluates accuracy only during geomagnetic storms ($D_{st}<-50$ nT), the primary region of interest. **MASE** expresses forecast skill relative to the Naive Persistence baseline, with values below 1 indicating an improvement over persistence. Finally, the **Diebold–Mariano (DM) test** assesses whether the difference in forecast accuracy between two competing models is statistically significant.

All models are evaluated using the same four metrics.

**Root Mean Squared Error (RMSE)** measures the overall prediction error:

$$
\mathrm{RMSE}=
\sqrt{\frac{1}{N}\sum_{i=1}^{N}(y_i-\hat{y}_i)^2},
$$

where $y_i$ is the observed $D_{st}$, $\hat{y}_i$ is the predicted value, and $N$ is the number of observations. Lower values indicate better overall accuracy.

**Storm RMSE** is computed using only storm observations ($D_{st}<-50$ nT):

$$
\mathrm{Storm\ RMSE}=
\sqrt{\frac{1}{N_{\rm storm}}
\sum_{D_{st}<-50}(y_i-\hat{y}_i)^2},
$$

where $N_{\rm storm}$ is the number of storm samples. This metric evaluates model performance during geomagnetic storms.

**Mean Absolute Scaled Error (MASE)** `[HYN06]` compares the prediction error with the average one-step error in the training series:

$$
\mathrm{MASE}=
\frac{\frac{1}{N}\sum |y_i-\hat{y}_i|}
{\frac{1}{N_{\rm train}-1}\sum |y_t-y_{t-1}|},
$$

where the denominator is computed from **Train₁**. Values below 1 indicate an improvement over the Naive Persistence baseline.

**Diebold–Mariano (DM) statistic** `[DM95]` tests whether two forecasting models have significantly different prediction accuracy:

$$
DM=
\frac{\bar d}
{\sqrt{\widehat{\mathrm{Var}}(\bar d)}},
$$

where $\bar d$ is the mean difference between the loss functions of the two models. A statistically significant negative value indicates that the evaluated model outperforms the reference model.

> **Reproducibility Note — `n_jobs` and Parallel Tree Construction**
>
> Every `XGBoostDst` fit in this notebook uses `n_jobs=N_JOBS_VALIDATION` (set to 1 in the first cell) rather than the class default of `n_jobs=-1`. This was adopted after an earlier version of this notebook, run twice with identical code and `random_state=42` but `n_jobs=-1`, produced different results between the two runs — including a sign change in $\Delta R^2$ at $h=12$h on `Val_Main`. Two independent controlled tests (against the actual project data, with both XGBoost 3.2.0 and 3.3.0) later confirmed that `XGBoostDst` itself is bit-reproducible regardless of `n_jobs` when the input data and library versions are held fixed — meaning the original instability came from the data/artifact pipeline, not from parallel tree construction. `n_jobs=1` is kept here regardless, since it was independently verified (three repeated executions for the cells above this note, two for the cells below it, all matching exactly) to produce fully reproducible results end-to-end for this specific notebook — the full investigation is documented in the Summary section at the end.

### Establishing Predictability

The first scientific question is whether future geomagnetic activity is predictable at all, and if so, what the primary source of predictability is. Three increasingly informative baseline models are therefore evaluated:

1. **Naive Persistence**, which assumes that future geomagnetic conditions remain unchanged.
2. **Autoregressive Persistence**, which predicts future $D_{st}$ from its current value alone.
3. **Solar Wind Forcing**, which predicts future $D_{st}$ from the selected OMNI solar wind parameters.

Comparing these baselines establishes the incremental predictive value contributed by progressively richer sources of information. Only after this baseline is established is the contribution of neutron monitor observations evaluated in the following section.

| Variable                 | Role in the scientific validation                                                                                                    | First used by                |
| ------------------------ | ------------------------------------------------------------------------------------------------------------------------------------ | ---------------------------- |
| `feat`                   | Complete feature matrix containing predictors, targets, and timestamps from which all training and validation subsets are extracted. | Naive Persistence            |
| `masks`                  | Defines the chronological validation segments (`Val_Main`, `Val_Storm`) used consistently by every experiment.                       | Naive Persistence            |
| `BOUNDARIES`, `PURGE_H`  | Reconstruct the `Train_1` reference segment and enforce purge-aware temporal boundaries for metric computation.                      | Naive Persistence            |
| `train1_mask`, `y_train` | Provides the reference training series required to compute the MASE denominator for every model comparison.                          | Naive Persistence            |
| `K_HORIZONS`             | Defines the five forecast horizons (1, 3, 7, 12, 21 h) evaluated throughout all experiments.                                         | All models                   |
| `STORM_THR`              | Common storm threshold ($D_{st}<-50$ nT) used for Storm RMSE and weighted evaluation.                                                | All models                   |
| `SELECTED_FEATURES`      | Final predictor subset obtained from Feature Screening and used to construct the machine-learning feature sets.                      | Solar Wind Forcing (XGBoost) |
| `scaler`                 | Standardises the selected predictors before fitting machine-learning models. Not required by the persistence baselines.              | Solar Wind Forcing (XGBoost) |


The cell below loads the shared context for this notebook: the feature matrix, the selected predictor subset and its fitted `StandardScaler` (both produced by `feature_selection.ipynb`), and the three masks used throughout — `train1` for fitting every model below, `val_main` and `val_storm` for evaluating all of them. Every result in this notebook traces back to these five objects.

In [2]:
# ── Scientific Validation Context ────────────────────────────────────────
feat_data         = pd.read_parquet('data/processed/feat_split.parquet')
fs                = joblib.load('models/feature_selection_results.pkl')
SELECTED_FEATURES = fs['selected_final']
scaler            = fs['scaler']
feature_sets      = build_feature_sets(SELECTED_FEATURES)

masks          = build_masks(feat_data['datetime'])
val_main_mask  = masks['val_main']
val_storm_mask = masks['val_storm']
train1_mask    = masks['train1']

y_train = feat_data.loc[train1_mask, 'dst'].copy()

EVAL_SEGMENTS = {
    'val_main' : val_main_mask,
    'val_storm': val_storm_mask,
}

print(f"SELECTED_FEATURES : {len(SELECTED_FEATURES)}")
print(f"K_HORIZONS        : {K_HORIZONS}")
print(f"STORM_THR         : {STORM_THR} nT")
print(f"Train reference   : {train1_mask.sum():,} rows")
print(f"Val_Main          : {val_main_mask.sum():,} rows")
print(f"Val_Storm         : {val_storm_mask.sum():,} rows")

SELECTED_FEATURES : 24
K_HORIZONS        : [1, 3, 7, 12, 21]
STORM_THR         : -50.0 nT
Train reference   : 76,995 rows
Val_Main          : 52,542 rows
Val_Storm         : 1,446 rows


The four predictor configurations used in the ablation study are constructed here. `OMNI_FEATURES` is derived from `SELECTED_FEATURES` by excluding all neutron-derived variables, providing the solar-wind-only baseline. The remaining configurations extend this baseline by adding neutron representations one at a time, following the A→B→C→D experimental sequence.

In [3]:
# ── Feature set definitions ───────────────────────────────────────────────

feature_sets      = build_feature_sets(SELECTED_FEATURES)
OMNI_FEATURES         = feature_sets['OMNI_FEATURES']
MODEL_A_OMNI          = feature_sets['MODEL_A_OMNI']
MODEL_B_OMNI_RAW      = feature_sets['MODEL_B_OMNI_RAW']
MODEL_C_OMNI_DNEUTRON = feature_sets['MODEL_C_OMNI_DNEUTRON']
MODEL_D_OMNI_HISTORY  = feature_sets['MODEL_D_OMNI_HISTORY']

print(f"OMNI features      : {len(MODEL_A_OMNI)}")
print(f"MODEL_B features   : {len(MODEL_B_OMNI_RAW)}")
print(f"MODEL_C features   : {len(MODEL_C_OMNI_DNEUTRON)}")
print(f"MODEL_D features   : {len(MODEL_D_OMNI_HISTORY)}")

OMNI features      : 20
MODEL_B features   : 21
MODEL_C features   : 21
MODEL_D features   : 23


#### Naive Persistence

The naïve persistence model assumes that the future geomagnetic state is identical to the current one:

$$
\hat{D}_{st}(t+h)=D_{st}(t).
$$

No learning is performed and no model parameters are estimated. The current observation is used directly as the prediction for every forecast horizon. Persistence therefore represents the minimum predictive baseline that any forecasting model must outperform.

For each forecast horizon $h \in \{1,3,7,12,21\}$, the forecast error is simply

$$
e_t^{(h)} = D_{st}(t+h)-D_{st}(t),
$$

which measures the natural evolution of the geomagnetic field over the prediction interval.

Persistence is evaluated independently on `Val_Main` (2009–2014) and `Val_Storm` (Halloween–November 2003). These results establish the reference RMSE, Storm RMSE, and MASE values against which all subsequent forecasting models are compared. By definition, the persistence model has $\mathrm{MASE}=1$, providing the natural reference point for forecast skill. Likewise, the Diebold–Mariano statistic is zero (p = 1.0) when persistence is compared with itself, serving as a sanity check for the evaluation pipeline.

The code below evaluates this baseline using the common validation setup defined above. `feat` provides both the current $D_{st}(t)$ values and the target columns `dst_target_{h}h`. `K_HORIZONS` ensures that the same five horizons are evaluated for every model. `EVAL_SEGMENTS` runs the test separately on `Val_Main` and `Val_Storm`, while `y_train` is passed only to compute the MASE reference scale. Since persistence is compared against itself, `y_persist_fn=None` and no additional persistence reference is required.

The persistence baseline requires no training, but it follows the same experimental protocol as all subsequent models. The reference series `y_train` is extracted from **Train1** only and is used to compute the MASE denominator, ensuring that forecast skill is measured relative to the same training period for every model. The baseline is then evaluated independently on `Val_Main` and `Val_Storm` across the five forecast horizons defined by `K_HORIZONS`.

Unlike persistence, the direct autoregressive baseline must be trained. Because the model predicts each forecast horizon directly rather than recursively, a separate linear model is fitted for every horizon in `K_HORIZONS`.

For horizon $h$, the model receives the current geomagnetic state `dst` as input and learns the mapping to the corresponding target `dst_target_{h}h` using **Train₁**. The fitted coefficients $(\alpha_h,\beta_h)$ therefore describe how the linear memory of the geomagnetic field changes with forecast horizon. The fitted coefficients are extracted for each horizon and stored separately. Their evolution with forecast horizon provides a direct interpretation of how rapidly the predictive influence of the current $D_{st}$ state decays.

The cell below evaluates `NaivePersistence` at every horizon in `K_HORIZONS`, on both `Val_Main` and `Val_Storm`, and saves the metrics for later comparison. No fitting happens — this is purely a measurement of how much the $D_{st}$ series changes over each forecast interval.

In [4]:
# ── Naive Persistence Evaluation ──────────────────────────────────────────
# Evaluates the persistence baseline on all forecast horizons and both
# validation segments. Results are saved for later comparison with the
# autoregressive and machine-learning models.

persistence_model = NaivePersistence(dst_col="dst")

persistence_metrics = {
    seg_name: run_segment(
        model_name="naive_persistence",
        seg_name=seg_name,
        models={h: persistence_model for h in K_HORIZONS},
        X_seg_fn=lambda h: feat_data.loc[seg_mask],
        y_true_fn=lambda h: feat_data.loc[seg_mask, f"dst_target_{h}h"],
        y_train=y_train,
        y_persist_fn=None,
        storm_thr=STORM_THR,
        k_horizons=K_HORIZONS,
    )
    for seg_name, seg_mask in EVAL_SEGMENTS.items()
}

joblib.dump(
    persistence_metrics,
    "models/metrics_naive_persistence.pkl"
)

print("Saved: models/metrics_naive_persistence.pkl")


── Segment: val_main ──────────────────────────────────────
  h= 1h | RMSE=  3.55 | StormRMSE=  9.19 | MASE=0.754
  h= 3h | RMSE=  7.37 | StormRMSE= 22.34 | MASE=1.587
  h= 7h | RMSE= 10.90 | StormRMSE= 38.85 | MASE=2.300
  h=12h | RMSE= 13.27 | StormRMSE= 50.46 | MASE=2.774
  h=21h | RMSE= 15.44 | StormRMSE= 60.25 | MASE=3.240

── Segment: val_storm ──────────────────────────────────────
  h= 1h | RMSE= 10.04 | StormRMSE= 24.41 | MASE=1.626
  h= 3h | RMSE= 22.79 | StormRMSE= 58.54 | MASE=3.372
  h= 7h | RMSE= 38.76 | StormRMSE=102.29 | MASE=5.378
  h=12h | RMSE= 47.71 | StormRMSE=126.01 | MASE=6.792
  h=21h | RMSE= 54.42 | StormRMSE=142.70 | MASE=7.997
Saved: models/metrics_naive_persistence.pkl


> **Observations — Naive Persistence.** This is the floor every subsequent model has to clear. At $h=1$h it is a genuinely hard floor to beat: MASE = 0.754 on `Val_Main`, meaning the current $D_{st}$ value alone already predicts one hour ahead better than the average one-step change in the training series. That advantage evaporates fast — by $h=7$h, Storm RMSE on `Val_Main` has grown from 9.19 to 38.85 nT, and on `Val_Storm` (the Halloween 2003 event) it reaches 102.29 nT. The gap between the two segments at every horizon — `Val_Storm` is consistently 2–3× worse than `Val_Main` — is itself informative: it quantifies how much harder the extreme event is to describe by assuming nothing changes. Whatever comes next needs to close that gap, not just beat persistence on average.

#### Direct Autoregressive Baseline

The direct autoregressive baseline predicts future geomagnetic activity using only the current geomagnetic state:

$$
\hat{D}_{st}(t+h)=\alpha_h D_{st}(t)+\beta_h.
$$

A separate linear regression model is fitted for each forecast horizon using **Train₁** only. Unlike recursive autoregressive forecasting, the model predicts $D_{st}(t+h)$ directly in a single step, avoiding the accumulation of intermediate prediction errors.

This baseline quantifies the predictive information contained in the current $D_{st}$ value alone. Compared with persistence, the coefficients $\alpha_h$ and $\beta_h$ are estimated from the data rather than fixed to $(1,0)$, allowing the model to learn the average decay of geomagnetic disturbances.

Only the current $D_{st}(t)$ is used as a predictor. No additional lags or solar wind variables are included, ensuring that any subsequent improvement can be attributed to external solar wind forcing rather than a more complex autoregressive model.

The model is trained on **Train₁** and evaluated independently on `Val_Main` and `Val_Storm` using the same forecast horizons, metrics, and evaluation protocol as the persistence baseline. The fitted coefficients $\alpha_h$ and $\beta_h$ are retained to examine how the contribution of the current geomagnetic state changes with forecast horizon.

| Argument                                    | Meaning                     | Why this model needs it                                                      |
| ------------------------------------------- | --------------------------- | ---------------------------------------------------------------------------- |
| `DirectARBaseline(dst_col='dst')`           | Linear autoregressive model | Learns the relationship between the current and future (D_{st}).             |
| `feat_data.loc[train1_mask, ['dst']]`            | Predictor (D_{st}(t))       | The model uses only the current geomagnetic state.                           |
| `feat_data.loc[train1_mask, f'dst_target_{h}h']` | Target (D_{st}(t+h))        | A separate model is trained for each forecast horizon.                       |
| `train1_mask`                               | Training subset             | Ensures all baselines are fitted only on **Train₁**.                         |
| `K_HORIZONS`                                | Forecast horizons           | Creates five independent autoregressive models (1, 3, 7, 12, 21 h).          |
| `EVAL_SEGMENTS`                             | Validation subsets          | Evaluates every model on `Val_Main` and `Val_Storm` using the same protocol. |
| `y_train`                                   | Reference training series   | Required only for computing the MASE denominator.                            |
| `y_persist_fn`                              | Persistence forecast        | Provides the reference forecast for the Diebold–Mariano comparison.          |
| `extra_params_fn`                           | Returns `α` and `β`         | Saves the fitted autoregressive coefficients for later interpretation.       |


A separate `DirectARBaseline` is fitted per horizon on `Train₁`, then evaluated the same way as persistence above — same horizons, same two segments, same metrics — so the two baselines are directly comparable.

In [5]:
# ── Fit one model per horizon (Train_1 only) ─────────────────────────────
ar_models = {
    h: fit_model(
        DirectARBaseline(dst_col='dst'),
        feat_data.loc[train1_mask, ['dst']],
        feat_data.loc[train1_mask, f'dst_target_{h}h'],
    )
    for h in K_HORIZONS
}

ar_coefs = {h: {'alpha': ar_models[h].alpha_, 'beta': ar_models[h].beta_}
            for h in K_HORIZONS}

# ── Evaluate on both validation segments ─────────────────────────────────
ar_metrics = {
    seg_name: run_segment(
        model_name      = 'direct_ar',
        seg_name        = seg_name,
        models          = ar_models,
        X_seg_fn        = lambda h: feat_data.loc[seg_mask, ['dst']],
        y_true_fn       = lambda h: feat_data.loc[seg_mask, f'dst_target_{h}h'],
        y_train         = y_train,
        y_persist_fn    = lambda h: feat_data.loc[seg_mask, 'dst'].values,
        storm_thr       = STORM_THR,
        k_horizons      = K_HORIZONS,
        extra_params_fn = lambda h, m: {
            'alpha': round(m.alpha_, 4),
            'beta' : round(m.beta_,  4),
        },
    )
    for seg_name, seg_mask in EVAL_SEGMENTS.items()
}

joblib.dump(ar_metrics, 'models/metrics_direct_ar.pkl')
print('\nSaved: models/metrics_direct_ar.pkl')


── Segment: val_main ──────────────────────────────────────
  alpha=0.977  beta=-0.380 | h= 1h | RMSE=  3.53 | StormRMSE=  9.26 | MASE=0.763
  alpha=0.902  beta=-1.612 | h= 3h | RMSE=  7.20 | StormRMSE= 22.49 | MASE=1.576
  alpha=0.774  beta=-3.731 | h= 7h | RMSE= 10.34 | StormRMSE= 38.08 | MASE=2.268
  alpha=0.660  beta=-5.622 | h=12h | RMSE= 12.25 | StormRMSE= 47.81 | MASE=2.727
  alpha=0.526  beta=-7.835 | h=21h | RMSE= 13.83 | StormRMSE= 54.98 | MASE=3.167

── Segment: val_storm ──────────────────────────────────────
  alpha=0.977  beta=-0.380 | h= 1h | RMSE= 10.00 | StormRMSE= 24.39 | MASE=1.605
  alpha=0.902  beta=-1.612 | h= 3h | RMSE= 22.26 | StormRMSE= 57.44 | MASE=3.157
  alpha=0.774  beta=-3.731 | h= 7h | RMSE= 36.15 | StormRMSE= 95.87 | MASE=4.747
  alpha=0.660  beta=-5.622 | h=12h | RMSE= 42.75 | StormRMSE=113.62 | MASE=5.761
  alpha=0.526  beta=-7.835 | h=21h | RMSE= 46.80 | StormRMSE=124.34 | MASE=6.555

Saved: models/metrics_direct_ar.pkl


> **Observations — Direct Autoregressive Baseline.** Letting the model learn the decay instead of assuming none ($\alpha=1$) already buys something: $\alpha$ falls from 0.977 at $h=1$h to 0.526 at $h=21$h, tracking the physical decay of the ring current rather than freezing $D_{st}$ in place. The payoff shows up exactly where persistence struggled — Storm RMSE on `Val_Storm` at $h=7$h drops from 102.29 nT (persistence) to 95.87 nT, a larger relative gain than the corresponding improvement on `Val_Main` (38.85 → 38.08 nT, barely moved). So the first piece of the story is already visible with a single scalar predictor: storm dynamics carry learnable structure that persistence throws away. The open question is whether that structure is fully captured by $D_{st}$'s own memory, or whether external drivers — solar wind, and eventually neutron flux — add something $D_{st}(t)$ alone cannot see.

#### Solar Wind Forcing (MODEL_A)

The previous experiment demonstrated the predictive information contained in the current geomagnetic state alone. The next question is whether the upstream solar wind provides additional predictive information beyond this linear autoregressive memory.

To answer this question, a nonlinear XGBoost model is trained using only the OMNI solar wind predictors:

$$
\hat{D}_{st}(t+h)=f_{\mathrm{XGB}}(\mathbf{x}_{\mathrm{OMNI}}(t)).
$$

Unlike the Direct AR baseline, which uses only the current $D_{st}$ value, XGBoost can model nonlinear relationships and interactions between multiple solar wind parameters. A separate model is trained for each forecast horizon using the 19 OMNI features retained during Feature Screening.

The machine-learning experiments are organised as a sequence of predefined feature sets. The same learning algorithm, training procedure, validation protocol, and evaluation metrics are used throughout. Only the predictor set changes between experiments, allowing the contribution of each additional source of information to be isolated.

- **MODEL_A** – OMNI solar wind features only.
- **MODEL_B** – OMNI features + raw neutron counts.
- **MODEL_C** – OMNI features + differential neutron flux ($\delta n$).
- **MODEL_D** – OMNI features + all selected neutron predictors.

In this experiment, **MODEL_A** establishes the nonlinear solar wind baseline. The remaining models will be compared against MODEL_A to determine whether neutron observations provide additional predictive information beyond the conventional OMNI measurements.

The models are trained on **Train₁** and evaluated on `Val_Main` and `Val_Storm` using the same protocol as the previous baselines. Storm sample weighting is applied during training to compensate for the severe imbalance between quiet and storm periods, ensuring that the model learns both regimes. Default XGBoost hyperparameters are used intentionally, as the objective is to evaluate the contribution of the predictor set rather than to maximise model performance. Hyperparameter optimisation is performed later, after the forecast horizon has been selected.

Before constructing the forecasting models, the selected predictors are grouped according to their physical origin. This allows the same modelling pipeline to evaluate progressively richer feature sets while changing only the neutron-related information.

- `NEUTRON_ALL` contains all neutron-monitor-derived predictors retained after Feature Screening.
- `NEUTRON_RAW` contains the instantaneous neutron count (`neutron_counts`).
- `NEUTRON_DERIVED` contains the differential neutron flux (`d_neutron`), the primary neutron representation investigated in this study.
- `NEUTRON_HISTORY` contains the lagged neutron observations representing the temporal evolution of the Forbush Decrease signal.

The remaining predictors form the conventional OMNI feature set:

- `OMNI_FEATURES` contains all selected predictors except the neutron-derived variables.

These groups are combined to construct the forecasting models used throughout the Scientific Validation stage. The first model,

- `MODEL_A_OMNI = OMNI_FEATURES`,

serves as the reference baseline containing only the conventional solar wind observations. Subsequent models progressively augment this baseline with neutron-derived predictors while leaving the learning algorithm, training procedure, and evaluation protocol unchanged.

A separate XGBoost model is fitted for each forecasting horizon using the `MODEL_A_OMNI` feature set defined in the Experimental Design. Each model is trained exclusively on **Train₁**, where the predictors are the selected OMNI features and the target is the corresponding future geomagnetic disturbance `dst_target_{h}h`.

The fitted models are subsequently evaluated on both validation segments (`Val_Main` and `Val_Storm`) using the common evaluation protocol established in the Experimental Design. The persistence forecast is supplied as the reference model for the Diebold–Mariano significance test, while the **Train₁** series is used to compute the MASE denominator.

For reproducibility, each fitted XGBoost pipeline is logged as an MLflow artifact and the evaluation metrics are stored in `metrics_xgb_model_a.pkl`, providing the reference nonlinear OMNI baseline for all subsequent neutron-feature experiments.

In [6]:
# ── Fit one pipeline per horizon (Train_1 only) ───────────────────────────
model_a_pipes = {
    h: fit_model(
        Pipeline([('model', XGBoostDst(n_jobs=N_JOBS_VALIDATION))]),
        feat_data.loc[train1_mask, MODEL_A_OMNI],
        feat_data.loc[train1_mask, f'dst_target_{h}h'],
    )
    for h in K_HORIZONS
}

# ── Evaluate on both validation segments ──────────────────────────────────
model_a_metrics = {
    seg_name: run_segment(
        model_name      = 'xgb_model_a_omni',
        seg_name        = seg_name,
        models          = model_a_pipes,
        X_seg_fn        = lambda h: feat_data.loc[seg_mask, MODEL_A_OMNI],
        y_true_fn       = lambda h: feat_data.loc[seg_mask, f'dst_target_{h}h'],
        y_train         = y_train,
        y_persist_fn    = lambda h: feat_data.loc[seg_mask, 'dst'].values,
        storm_thr       = STORM_THR,
        k_horizons      = K_HORIZONS,
        extra_params_fn = lambda h, m: {
            'feature_set': 'MODEL_A_OMNI',
            'n_features' : len(MODEL_A_OMNI),
        },
    )
    for seg_name, seg_mask in EVAL_SEGMENTS.items()
}

joblib.dump(model_a_metrics, 'models/metrics_xgb_model_a.pkl')
print('\nSaved: models/metrics_xgb_model_a.pkl')


── Segment: val_main ──────────────────────────────────────
  h= 1h | RMSE= 10.92 | StormRMSE= 18.85 | MASE=2.705
  h= 3h | RMSE= 11.55 | StormRMSE= 20.32 | MASE=2.831
  h= 7h | RMSE= 14.33 | StormRMSE= 29.03 | MASE=3.420
  h=12h | RMSE= 17.00 | StormRMSE= 39.53 | MASE=3.941
  h=21h | RMSE= 19.17 | StormRMSE= 49.85 | MASE=4.674

── Segment: val_storm ──────────────────────────────────────
  h= 1h | RMSE= 30.45 | StormRMSE= 79.32 | MASE=4.513
  h= 3h | RMSE= 33.18 | StormRMSE= 85.65 | MASE=4.911
  h= 7h | RMSE= 40.17 | StormRMSE=104.87 | MASE=5.607
  h=12h | RMSE= 44.45 | StormRMSE=116.38 | MASE=6.285
  h=21h | RMSE= 50.14 | StormRMSE=131.51 | MASE=7.214

Saved: models/metrics_xgb_model_a.pkl


> **Observations — XGBoost MODEL_A (OMNI only).** Bringing in the solar wind driver, and letting a nonlinear model use it, moves the needle on storms specifically: Storm RMSE on `Val_Main` at $h=7$h falls further, from AR's 38.08 nT to 29.03 nT — a bigger single jump than AR's own improvement over persistence. But this comes at a cost the simpler baselines didn't pay: overall MASE stays above 1 at every horizon (e.g. 3.420 at $h=7$h), so on average absolute error MODEL_A is still behind persistence — an untuned, default-hyperparameter model is not yet competitive on typical hours, only on the tail it was built to capture. On `Val_Storm` the picture reverses again: MODEL_A's Storm RMSE at $h=7$h (104.87 nT) is *worse* than both persistence (102.29 nT) and AR (95.87 nT) — the Halloween 2003 event, at $D_{st}=-422$ nT, sits far enough outside anything MODEL_A saw in `Train₁` that its solar-wind-driven nonlinearity does not yet generalise there. This is the baseline the neutron-augmented models now have to beat on both fronts — routine storms (`Val_Main`) and the one genuinely extreme event (`Val_Storm`) — not just on average.

### Neutron Flux as a Predictive Signal

The primary scientific objective of this study is to determine whether neutron-monitor-derived features provide additional predictive information beyond the conventional OMNI solar wind parameters for multi-hour forecasting of the $D_{st}$ index. This section addresses that objective through a controlled comparison between equivalent models with and without neutron-derived predictors.

To ensure a fair comparison, all models are evaluated using the common experimental protocol established in the previous section, including identical chronological data splits, forecasting horizons, model architecture and evaluation metrics. Consequently, any observed differences in predictive performance can be attributed to the inclusion and representation of neutron-derived features rather than to differences in the modelling approach.

The section evaluates the proposed differential neutron flux model (MODEL_C), then compares alternative neutron feature representations through a controlled ablation study, and concludes with a feature importance analysis that quantifies the contribution of neutron-derived features across all forecasting horizons.

#### Experimental Design

The research objective is investigated through a controlled comparison of two equivalent supervised regression models. For each forecasting horizon $h$, both models estimate the future geomagnetic disturbance index

\hat{D}_{st}(t+h) = f(\mathbf{x}(t)),\hat{D}_{st}(t+h) = f(\mathbf{x}(t)),

where $\mathbf{x}(t)$ denotes the predictor vector available at time $t$. The learning algorithm, chronological training and validation splits, forecasting horizons and evaluation protocol remain identical. The only experimental variable is the composition of the predictor vector:

$$\begin{aligned}
\text{MODEL}_A &: \mathbf{x}(t) = \mathbf{x}_{\mathrm{OMNI}}(t), \\
\text{MODEL}_C &: \mathbf{x}(t) = \{\mathbf{x}_{\mathrm{OMNI}}(t),\, \delta n(t)\},
\end{aligned}$$

where $\delta n(t)$ is the differential neutron flux. By modifying only the predictor representation while keeping all other modelling components unchanged, the contribution of $\delta n(t)$ can be evaluated independently of the learning algorithm.

The implementation follows the common experimental protocol:

- **Training data:** Train₁ chronological segment.
- **Forecasting horizons:** $h \in \{1, 3, 7, 12, 21\}$ hours.
- **Learning algorithm:** XGBoost regressor with default hyperparameters.
- **Predictor set:** `MODEL_C_OMNI_DNEUTRON` — OMNI solar wind features augmented with $\delta n(t)$.
- **Target variable:** $D_{st}(t+h)$.
- **Evaluation segments:** `Val_Main` and `Val_Storm`.
- **Evaluation metrics:** $R^2$, RMSE, Storm RMSE and MASE.

In [7]:
MODEL_C_OMNI_DNEUTRON = OMNI_FEATURES + NEUTRON_DERIVED

# ── Fit one pipeline per forecasting horizon using Train_1 ────────────────
model_c_pipes = {
    h: fit_model(
        Pipeline([('model', XGBoostDst(n_jobs=N_JOBS_VALIDATION))]),
        feat_data.loc[train1_mask, MODEL_C_OMNI_DNEUTRON],
        feat_data.loc[train1_mask, f'dst_target_{h}h'],
    )
    for h in K_HORIZONS
}

# ── Evaluate on the validation segments ───────────────────────────────────
model_c_metrics = {
    seg_name: run_segment(
        model_name      = 'xgb_model_c_omni_dneutron',
        seg_name        = seg_name,
        models          = model_c_pipes,
        X_seg_fn        = lambda h, seg_mask=seg_mask: feat_data.loc[seg_mask, MODEL_C_OMNI_DNEUTRON],
        y_true_fn       = lambda h, seg_mask=seg_mask: feat_data.loc[seg_mask, f'dst_target_{h}h'],
        y_train         = y_train,
        y_persist_fn    = lambda h, seg_mask=seg_mask: feat_data.loc[seg_mask, 'dst'].values,
        storm_thr       = STORM_THR,
        k_horizons      = K_HORIZONS,
        extra_params_fn = lambda h, m: {
            'feature_set': 'MODEL_C_OMNI_DNEUTRON',
            'n_features' : len(MODEL_C_OMNI_DNEUTRON),
        },
    )
    for seg_name, seg_mask in EVAL_SEGMENTS.items()
}

joblib.dump(model_c_metrics, 'models/metrics_xgb_model_c.pkl')
print('\nSaved: models/metrics_xgb_model_c.pkl')


── Segment: val_main ──────────────────────────────────────
  h= 1h | RMSE= 10.86 | StormRMSE= 19.08 | MASE=2.683
  h= 3h | RMSE= 11.56 | StormRMSE= 20.50 | MASE=2.828
  h= 7h | RMSE= 13.88 | StormRMSE= 29.18 | MASE=3.327
  h=12h | RMSE= 16.73 | StormRMSE= 40.00 | MASE=3.879
  h=21h | RMSE= 19.29 | StormRMSE= 49.05 | MASE=4.695

── Segment: val_storm ──────────────────────────────────────
  h= 1h | RMSE= 30.05 | StormRMSE= 78.10 | MASE=4.483
  h= 3h | RMSE= 32.28 | StormRMSE= 83.03 | MASE=4.838
  h= 7h | RMSE= 39.14 | StormRMSE=101.32 | MASE=5.624
  h=12h | RMSE= 43.90 | StormRMSE=114.65 | MASE=6.242
  h=21h | RMSE= 48.97 | StormRMSE=128.03 | MASE=7.149

Saved: models/metrics_xgb_model_c.pkl


> **Observations — XGBoost MODEL_C (OMNI + $\delta n$).** Adding a single derived feature moves both numbers from MODEL_A, and in the direction the research question hopes for: Storm RMSE at $h=7$h improves on `Val_Main` (29.03 → 29.18 nT — negligible, within noise) but more meaningfully on `Val_Storm` (104.87 → 101.32 nT). That the improvement shows up specifically on the extreme event, not on the routine one, is the first direct hint that $\delta n(t)$ carries information tied to the physical mechanism it was built for — a Forbush Decrease precursor that only manifests during genuinely large disturbances. The next cell checks whether the model is actually *using* this feature, or whether the RMSE shift is incidental.

`d_neutron`'s rank and gain share within MODEL_C's own feature set, extracted directly from each fitted XGBoost booster — a direct check of whether the RMSE improvement above reflects the model actually splitting on this feature, rather than an artifact of a different random tree structure.

In [8]:
# ── d_neutron feature importance по всички хоризонти ─────────────────────
print(f"{'h':>4} {'d_neutron rank':>16} {'gain %':>8} {'total features':>16}")
print("─" * 48)
for h in K_HORIZONS:
    booster = model_c_pipes[h].named_steps['model'].model_.get_booster()
    importance = booster.get_score(importance_type='gain')
    imp_df = pd.DataFrame.from_dict(importance, orient='index', columns=['gain'])
    imp_df = imp_df.sort_values('gain', ascending=False).reset_index()
    imp_df.columns = ['feature', 'gain']
    imp_df['rank'] = imp_df.index + 1
    imp_df['gain_pct'] = 100 * imp_df['gain'] / imp_df['gain'].sum()
    d = imp_df[imp_df['feature'] == 'd_neutron']
    if len(d) > 0:
        print(f"{h:>4} {d['rank'].values[0]:>16} {d['gain_pct'].values[0]:>8.2f}% {len(imp_df):>16}")
    else:
        print(f"{h:>4} {'NOT PRESENT':>16} {'0.00':>8}% {len(imp_df):>16}")

   h   d_neutron rank   gain %   total features
────────────────────────────────────────────────
   1               21     0.32%               21
   3               17     1.03%               21
   7               21     1.28%               21
  12               21     1.14%               21
  21               21     1.87%               21


#### Differential Neutron Flux Model

The predictive contribution of the differential neutron flux is evaluated by comparing MODEL_C with the OMNI-only reference model (MODEL_A). Since both models use the same learning algorithm, training procedure, forecasting horizons and evaluation protocol, the comparison isolates the effect of including the differential neutron flux in the predictor set.

The relative change in predictive performance is quantified as

$$\Delta R^2(h) = R^2(\text{MODEL\_C}) - R^2(\text{MODEL\_A})\Delta R^2(h) = R^2(\text{MODEL\_C}) - R^2(\text{MODEL\_A})$$

where positive values indicate improved predictive performance after augmenting the OMNI predictor set with the differential neutron flux.

The comparison is performed for all forecasting horizons and for both validation segments (`Val_Main` and `Val_Storm`).

In [9]:
def get_model_metrics(metrics_dict, seg_name, h):
    """Extract RMSE and Storm RMSE for one model/segment/horizon."""
    m = metrics_dict[seg_name][h]
    return round(m['rmse'], 2), round(m['storm_rmse'], 2)

def build_comparison_row(seg_name, h):
    """
    Build one comparison row for horizon h across all four models.
    Returns dict ready for DataFrame construction.
    """
    p_rmse,  p_srmse  = get_model_metrics(persistence_metrics, seg_name, h)
    ar_rmse, ar_srmse = get_model_metrics(ar_metrics,          seg_name, h)
    a_rmse,  a_srmse  = get_model_metrics(model_a_metrics,     seg_name, h)
    c_rmse,  c_srmse  = get_model_metrics(model_c_metrics,     seg_name, h)

    return {
        'h'                    : h,
        'Persistence RMSE'     : p_rmse,
        'Persistence StormRMSE': p_srmse,
        'AR RMSE'              : ar_rmse,
        'AR StormRMSE'         : ar_srmse,
        'XGB-A RMSE'           : a_rmse,
        'XGB-A StormRMSE'      : a_srmse,
        'XGB-C RMSE'           : c_rmse,
        'XGB-C StormRMSE'      : c_srmse,
    }

def print_comparison_table(seg_name):
    """Build and print comparison table for one segment."""
    rows = [build_comparison_row(seg_name, h) for h in K_HORIZONS]
    df   = pd.DataFrame(rows).set_index('h')
    print(f'\n── {seg_name} ──────────────────────────────────────')
    print(df.to_string())

for seg_name in EVAL_SEGMENTS:
    print_comparison_table(seg_name)


── val_main ──────────────────────────────────────
    Persistence RMSE  Persistence StormRMSE  AR RMSE  AR StormRMSE  XGB-A RMSE  XGB-A StormRMSE  XGB-C RMSE  XGB-C StormRMSE
h                                                                                                                           
1               3.55                   9.19     3.53          9.26       10.92            18.85       10.86            19.08
3               7.37                  22.34     7.20         22.49       11.55            20.32       11.56            20.50
7              10.90                  38.85    10.34         38.08       14.33            29.03       13.88            29.18
12             13.27                  50.46    12.25         47.81       17.00            39.53       16.73            40.00
21             15.44                  60.25    13.83         54.98       19.17            49.85       19.29            49.05

── val_storm ──────────────────────────────────────
    Persistence RMSE

> **Observations — Comparison of Forecasting Models.** Laid side by side, the four models trace a coherent progression at $h=7$h on `Val_Storm`: persistence 102.29 nT → AR 95.87 nT → MODEL_A 104.87 nT → MODEL_C 101.32 nT. MODEL_A's regression relative to AR is real and already flagged above; MODEL_C partially recovers it but does not yet beat AR outright at this horizon. On `Val_Main`, all four sit close together in overall RMSE (10.90–14.33 nT), confirming that the differences between models are concentrated in the storm tail, not in everyday behaviour — exactly where a Forbush-Decrease-motivated feature should matter, if it matters at all. `d_neutron`'s own rank (Cell 32: 21st of 21 at most horizons) sits in tension with this RMSE movement — the feature contributes little individually while the model's aggregate storm performance still shifts. $\Delta R^2$, computed next, is a better lens for this than RMSE alone, since it is normalised against the variance actually present in each segment.

The coefficient of determination ($R^2$) is used to compare the explanatory performance of MODEL_A and MODEL_C across all forecasting horizons. The corresponding $\Delta R^2$ values are computed directly from the stored evaluation results for each validation segment, providing a compact summary of the relative predictive contribution of the differential neutron flux.

In [10]:
def compute_delta_r2(seg_name, h):
    """
    Extract the coefficient of determination (R²) for MODEL_A and MODEL_C
    together with their difference (ΔR²) for a given validation segment
    and forecasting horizon.
    """
    r2_a = model_a_metrics[seg_name][h]['r2']
    r2_c = model_c_metrics[seg_name][h]['r2']
    return r2_a, r2_c, r2_c - r2_a


def print_delta_r2_table(seg_name):
    """
    Print the R² comparison table for all forecasting horizons within
    the selected validation segment.
    """
    print(f'\nΔR² = R²(OMNI+δn) - R²(OMNI)  [{seg_name}]')
    print('=' * 45)
    print(f'{"h":>4} | {"R²(A)":>8} | {"R²(C)":>8} | {"ΔR²":>8}')
    print('-' * 45)
    for h in K_HORIZONS:
        r2_a, r2_c, delta = compute_delta_r2(seg_name, h)
        print(f'{h:>4}h | {r2_a:>8.4f} | {r2_c:>8.4f} | {delta:>+8.4f}')

for seg_name in EVAL_SEGMENTS:
    print_delta_r2_table(seg_name)



ΔR² = R²(OMNI+δn) - R²(OMNI)  [val_main]
   h |    R²(A) |    R²(C) |      ΔR²
---------------------------------------------
   1h |   0.5092 |   0.5137 |  +0.0045
   3h |   0.4508 |   0.4493 |  -0.0015
   7h |   0.1544 |   0.2058 |  +0.0514
  12h |  -0.1900 |  -0.1529 |  +0.0372
  21h |  -0.5141 |  -0.5328 |  -0.0187

ΔR² = R²(OMNI+δn) - R²(OMNI)  [val_storm]
   h |    R²(A) |    R²(C) |      ΔR²
---------------------------------------------
   1h |   0.6226 |   0.6325 |  +0.0099
   3h |   0.5519 |   0.5759 |  +0.0240
   7h |   0.3432 |   0.3763 |  +0.0332
  12h |   0.1958 |   0.2156 |  +0.0198
  21h |  -0.0226 |   0.0246 |  +0.0471


> **Observations — Relative Performance ($\Delta R^2$).** On `Val_Main`, $\Delta R^2$ is positive at three of five horizons (+0.0045 at $h=1$h, +0.0514 at $h=7$h, +0.0372 at $h=12$h) and negative at the other two ($-0.0015$ at $h=3$h, $-0.0187$ at $h=21$h) — not a consistent trend, but the largest positive value on this segment (+0.0514) lands at $h=7$h, the same horizon already flagged for its RMSE movement. On `Val_Storm`, the pattern is cleaner: $\Delta R^2$ is positive at every horizon (+0.0099 to +0.0471), peaking at $h=21$h (+0.0471) with $h=7$h close behind (+0.0332). Two things follow from this. First, $\delta n(t)$'s contribution on `Val_Storm` is not a one-horizon coincidence — it holds across the full range tested. Second, $h=7$h is the only horizon where both segments show a positive, non-trivial $\Delta R^2$ simultaneously (+0.0514 and +0.0332) — every other horizon has at least one segment near zero or negative. That combination — not the single largest number on either segment alone — is what the Ablation Study and the significance tests below need to confirm or overturn.

#### Ablation Study

The ablation study investigates the predictive contribution of neutron-monitor information by separating three independent questions:

1. **Does neutron information provide additional predictive value beyond OMNI?** MODEL_A → MODEL_B introduces the raw neutron count rate $n(t)$ while keeping all other features unchanged.
2. **Which neutron representation captures the signal more effectively?** MODEL_B → MODEL_C compares the raw count rate with the differential flux $\delta n(t)$, isolating the effect of the feature transformation.
3. **Does neutron history provide additional predictive value beyond the current signal?** MODEL_C → MODEL_D evaluates whether lagged neutron counts ($n_{t-3h}$, $n_{t-7h}$) add information beyond the current differential flux.

This sequential design isolates one experimental factor at a time. MODEL_A and MODEL_C results are carried forward from the preceding experiments. MODEL_B and MODEL_D are trained in this section under the same XGBoost configuration, training split and evaluation protocol — the neutron representation is the only difference between the four configurations.

In [11]:
# ── Cell: Ablation Study — MODEL_B and MODEL_D ───────────────────────────
# MODEL_A and MODEL_C metrics already computed in Stages 3-4.
# Only MODEL_B (OMNI + raw neutron) and MODEL_D (OMNI + δn + lags) are new.

# ── Fit MODEL_B (OMNI + raw neutron counts) ──────────────────────────────
print('\n── MODEL_B: OMNI + raw neutron counts ───────────────────────────────')
model_b_pipes = {
    h: fit_model(
        Pipeline([('model', XGBoostDst(n_jobs=N_JOBS_VALIDATION))]),
        feat_data.loc[train1_mask, MODEL_B_OMNI_RAW],
        feat_data.loc[train1_mask, f'dst_target_{h}h'],
    )
    for h in K_HORIZONS
}

model_b_metrics = {
    seg_name: run_segment(
        model_name      = 'xgb_model_b_omni_raw',
        seg_name        = seg_name,
        models          = model_b_pipes,
        X_seg_fn        = lambda h: feat_data.loc[seg_mask, MODEL_B_OMNI_RAW],
        y_true_fn       = lambda h: feat_data.loc[seg_mask, f'dst_target_{h}h'],
        y_train         = y_train,
        y_persist_fn    = lambda h: feat_data.loc[seg_mask, 'dst'].values,
        storm_thr       = STORM_THR,
        k_horizons      = K_HORIZONS,
        extra_params_fn = lambda h, m: {
            'feature_set': 'MODEL_B_OMNI_RAW',
            'n_features' : len(MODEL_B_OMNI_RAW),
        },
    )
    for seg_name, seg_mask in EVAL_SEGMENTS.items()
}

joblib.dump(model_b_metrics, 'models/metrics_xgb_model_b.pkl')
print('Saved: models/metrics_xgb_model_b.pkl')


── MODEL_B: OMNI + raw neutron counts ───────────────────────────────

── Segment: val_main ──────────────────────────────────────
  h= 1h | RMSE= 11.33 | StormRMSE= 19.50 | MASE=2.824
  h= 3h | RMSE= 12.67 | StormRMSE= 19.90 | MASE=3.059
  h= 7h | RMSE= 14.67 | StormRMSE= 29.72 | MASE=3.473
  h=12h | RMSE= 16.57 | StormRMSE= 41.04 | MASE=3.827
  h=21h | RMSE= 17.63 | StormRMSE= 50.98 | MASE=4.199

── Segment: val_storm ──────────────────────────────────────
  h= 1h | RMSE= 30.35 | StormRMSE= 79.20 | MASE=4.528
  h= 3h | RMSE= 32.58 | StormRMSE= 82.91 | MASE=5.140
  h= 7h | RMSE= 39.59 | StormRMSE=101.31 | MASE=6.000
  h=12h | RMSE= 44.11 | StormRMSE=110.91 | MASE=7.335
  h=21h | RMSE= 50.43 | StormRMSE=128.94 | MASE=8.174
Saved: models/metrics_xgb_model_b.pkl


MODEL_D adds the two lagged neutron features that survived Feature Screening (`neutron_counts_lag3`, `neutron_counts_lag12`) on top of $\delta n(t)$, completing the third ablation question — does neutron *history* add anything beyond the current differential flux.

In [12]:
# ── Fit MODEL_D (OMNI + δn + neutron lags) ───────────────────────────────
print('\n── MODEL_D: OMNI + δn + neutron lags ───────────────────────────────')
model_d_pipes = {
    h: fit_model(
        Pipeline([('model', XGBoostDst(n_jobs=N_JOBS_VALIDATION))]),
        feat_data.loc[train1_mask, MODEL_D_OMNI_HISTORY],
        feat_data.loc[train1_mask, f'dst_target_{h}h'],
    )
    for h in K_HORIZONS
}

model_d_metrics = {
    seg_name: run_segment(
        model_name      = 'xgb_model_d_full_neutron',
        seg_name        = seg_name,
        models          = model_d_pipes,
        X_seg_fn        = lambda h: feat_data.loc[seg_mask, MODEL_D_OMNI_HISTORY],
        y_true_fn       = lambda h: feat_data.loc[seg_mask, f'dst_target_{h}h'],
        y_train         = y_train,
        y_persist_fn    = lambda h: feat_data.loc[seg_mask, 'dst'].values,
        storm_thr       = STORM_THR,
        k_horizons      = K_HORIZONS,
        extra_params_fn = lambda h, m: {
            'feature_set': 'MODEL_D_OMNI_HISTORY',
            'n_features' : len(MODEL_D_OMNI_HISTORY),
        },
    )
    for seg_name, seg_mask in EVAL_SEGMENTS.items()
}

joblib.dump(model_d_metrics, 'models/metrics_xgb_model_d.pkl')
print('Saved: models/metrics_xgb_model_d.pkl')


── MODEL_D: OMNI + δn + neutron lags ───────────────────────────────

── Segment: val_main ──────────────────────────────────────
  h= 1h | RMSE= 11.19 | StormRMSE= 18.96 | MASE=2.776
  h= 3h | RMSE= 11.93 | StormRMSE= 19.36 | MASE=2.914
  h= 7h | RMSE= 13.52 | StormRMSE= 29.31 | MASE=3.286
  h=12h | RMSE= 16.27 | StormRMSE= 41.15 | MASE=3.767
  h=21h | RMSE= 17.14 | StormRMSE= 52.90 | MASE=4.049

── Segment: val_storm ──────────────────────────────────────
  h= 1h | RMSE= 30.66 | StormRMSE= 80.09 | MASE=4.519
  h= 3h | RMSE= 32.76 | StormRMSE= 83.19 | MASE=5.145
  h= 7h | RMSE= 39.31 | StormRMSE=101.01 | MASE=5.828
  h=12h | RMSE= 44.51 | StormRMSE=113.93 | MASE=6.865
  h=21h | RMSE= 50.74 | StormRMSE=126.72 | MASE=8.680
Saved: models/metrics_xgb_model_d.pkl


The table below puts all four models' Storm RMSE side by side, then computes each neutron-augmented model's improvement over MODEL_A directly (positive = better than MODEL_A).

In [13]:
# ── Ablation Comparison — Storm RMSE, all models and horizons ─────────────
MODELS = {
    'A': model_a_metrics,
    'B': model_b_metrics,
    'C': model_c_metrics,
    'D': model_d_metrics,
}

SEGMENTS = {'storm': 'val_storm', 'quiet': 'val_main'}

def get_segment_metrics(metrics, seg, metric_key):
    return [metrics[seg][h][metric_key] for h in K_HORIZONS]

def get_model_comparison(label, metrics, metric_key):
    return {
        f'{label}_{col}': get_segment_metrics(metrics, seg, metric_key)
        for col, seg in SEGMENTS.items()
    }

def build_comparison(metric_key='storm_rmse'):
    data = {}
    for label, metrics in MODELS.items():
        data.update(get_model_comparison(label, metrics, metric_key))
    return pd.DataFrame(data, index=K_HORIZONS)

def delta_vs_a(metrics, h, seg):
    """Storm RMSE improvement vs MODEL_A."""
    return model_a_metrics[seg][h]['storm_rmse'] - metrics[seg][h]['storm_rmse']

storm_rmse_table = build_comparison('storm_rmse')
print("── Storm RMSE — all models, both segments ───────────────────────────")
print(storm_rmse_table.round(2))
print()

print(f"{'h':>4} {'B storm Δ':>10} {'B quiet Δ':>10} {'C storm Δ':>10} {'C quiet Δ':>10} {'D storm Δ':>10} {'D quiet Δ':>10}")
print("─" * 68)
for h in K_HORIZONS:
    vals = [(delta_vs_a(MODELS[m], h, 'val_storm'), 
             delta_vs_a(MODELS[m], h, 'val_main')) for m in ['B', 'C', 'D']]
    flat = [v for pair in vals for v in pair]
    print(f"{h:>4} " + " ".join(f"{v:>+10.2f}" for v in flat))

joblib.dump({'storm_rmse_table': storm_rmse_table}, 'models/ablation_comparison.pkl')
print('\nSaved: models/ablation_comparison.pkl')

── Storm RMSE — all models, both segments ───────────────────────────
    A_storm  A_quiet  B_storm  B_quiet  C_storm  C_quiet  D_storm  D_quiet
1     79.32    18.85    79.20    19.50    78.10    19.08    80.09    18.96
3     85.65    20.32    82.91    19.90    83.03    20.50    83.19    19.36
7    104.87    29.03   101.31    29.72   101.32    29.18   101.01    29.31
12   116.38    39.53   110.91    41.04   114.65    40.00   113.93    41.15
21   131.51    49.85   128.94    50.98   128.03    49.05   126.72    52.90

   h  B storm Δ  B quiet Δ  C storm Δ  C quiet Δ  D storm Δ  D quiet Δ
────────────────────────────────────────────────────────────────────
   1      +0.11      -0.64      +1.21      -0.23      -0.78      -0.11
   3      +2.75      +0.42      +2.62      -0.18      +2.46      +0.95
   7      +3.56      -0.69      +3.55      -0.15      +3.86      -0.28
  12      +5.47      -1.51      +1.73      -0.46      +2.45      -1.61
  21      +2.57      -1.14      +3.48      +0.80      +

> **Observations — Ablation Study.** At $h=7$h — the horizon `Val_Storm`'s $\Delta R^2$ and `Val_Main`'s $\Delta R^2$ both flagged above — all three neutron representations now agree: MODEL_B +3.56 nT, MODEL_C +3.55 nT, MODEL_D +3.86 nT of Storm RMSE improvement over MODEL_A on `Val_Storm`. The three representations converging this tightly at exactly the horizon flagged independently by $\Delta R^2$ is the first result in this notebook that does not depend on which specific neutron feature is chosen — raw count, differential flux, or history all move together. Away from $h=7$h the picture is less orderly: MODEL_D swings from $-0.78$ nT at $h=1$h to $+4.79$ nT at $h=21$h with no monotonic trend, and MODEL_B peaks unexpectedly at $h=12$h (+5.47 nT) rather than at the longest horizon. On `Val_Main`'s corresponding storm hours (the `_quiet` columns, computed with the identical Storm RMSE formula but within `Val_Main`), every model except MODEL_B at $h=21$h costs something (−0.11 to −3.05 nT) — meaning whatever is gained on the single extreme event in `Val_Storm` is not free elsewhere. Whether that trade-off is real or noise is exactly what the significance tests below are for; a point estimate this size, on an evaluation window of a single storm, is not yet evidence on its own.

### Feature Importance Summary

Gain-based feature importance is extracted from the fitted XGBoost models for all four predictor configurations across all five forecasting horizons. Gain measures the average improvement in the loss function when a feature is used in a tree split — a feature with near-zero gain is rarely selected and contributes minimally to the model's predictions regardless of its presence in the feature set.

This analysis serves two purposes: it quantifies whether neutron-derived features are actively used by the model, and it identifies the forecasting horizon at which the neutron signal provides the greatest relative contribution compared with the dominant OMNI features.

The table below shows gain-based feature importance for the neutron-derived features across all four models and five forecasting horizons. Gain measures the average improvement in the loss function when a feature is used in a tree split. The equal-share baseline (~4.76% for MODEL_B/C, ~4.35% for MODEL_D) represents the expected gain if all features contributed equally — values below this indicate the feature is used less than average.

In [14]:
# ── Cross table: neutron gain % по модел и хоризонт ──────────────────────
def get_gain_pct(pipes, h, feature):
    """Return gain % for a single feature in a model at horizon h."""
    booster = pipes[h].named_steps['model'].model_.get_booster()
    imp = pd.DataFrame.from_dict(
        booster.get_score(importance_type='gain'),
        orient='index', columns=['gain']
    )
    imp['gain_pct'] = 100 * imp['gain'] / imp['gain'].sum()
    d = imp[imp.index == feature]
    return round(d['gain_pct'].values[0], 2) if len(d) > 0 else 0.0


cross = pd.DataFrame({
    'MODEL_B (neutron_counts)' : [get_gain_pct(model_b_pipes, h, 'neutron_counts') for h in K_HORIZONS],
    'MODEL_C (d_neutron)'      : [get_gain_pct(model_c_pipes, h, 'd_neutron') for h in K_HORIZONS],
    'MODEL_D (lag3)'           : [get_gain_pct(model_d_pipes, h, 'neutron_counts_lag3') for h in K_HORIZONS],
    'MODEL_D (lag7)'           : [get_gain_pct(model_d_pipes, h, 'neutron_counts_lag7') for h in K_HORIZONS],
}, index=K_HORIZONS)
cross.index.name = 'h'


print("── Neutron feature gain % by model and horizon ──────────────────────")
print(f"{'':>4} {'MODEL_B':>20} {'MODEL_C':>15} {'MODEL_D lag3':>14} {'MODEL_D lag7':>14}")
print(f"{'h':>4} {'neutron_counts':>20} {'d_neutron':>15} {'lag3':>14} {'lag7':>14}")
print("─" * 67)
for h in [1, 3, 7, 12, 21]:
    print(f"{h:>4} {cross.loc[h,'MODEL_B (neutron_counts)']:>20.2f}% "
          f"{cross.loc[h,'MODEL_C (d_neutron)']:>14.2f}% "
          f"{cross.loc[h,'MODEL_D (lag3)']:>13.2f}% "
          f"{cross.loc[h,'MODEL_D (lag7)']:>13.2f}%")


── Neutron feature gain % by model and horizon ──────────────────────
                  MODEL_B         MODEL_C   MODEL_D lag3   MODEL_D lag7
   h       neutron_counts       d_neutron           lag3           lag7
───────────────────────────────────────────────────────────────────
   1                 1.13%           0.32%          1.09%          0.00%
   3                 1.26%           1.03%          0.94%          0.00%
   7                 1.87%           1.28%          1.77%          0.00%
  12                 3.61%           1.14%          3.55%          0.00%
  21                 6.89%           1.87%          7.26%          0.00%


The second table contextualises the neutron feature gain relative to the dominant OMNI predictor at each horizon. This shows how much of the total gain is captured by the top OMNI feature compared with the best neutron representation in each model.

In [15]:
def get_gain_pct(pipes, h, feature):
    booster = pipes[h].named_steps['model'].model_.get_booster()
    imp = pd.DataFrame.from_dict(
        booster.get_score(importance_type='gain'),
        orient='index', columns=['gain']
    )
    imp['gain_pct'] = 100 * imp['gain'] / imp['gain'].sum()
    d = imp[imp.index == feature]
    return round(d['gain_pct'].values[0], 2) if len(d) > 0 else 0.0

def get_top_omni(pipes, h):
    booster = pipes[h].named_steps['model'].model_.get_booster()
    imp = pd.DataFrame.from_dict(
        booster.get_score(importance_type='gain'),
        orient='index', columns=['gain']
    )
    imp['gain_pct'] = 100 * imp['gain'] / imp['gain'].sum()
    imp = imp.sort_values('gain_pct', ascending=False)
    top = imp.iloc[0]
    return top.name, top['gain_pct']

def get_equal_share(pipes, h):
    booster = pipes[h].named_steps['model'].model_.get_booster()
    return 100 / len(booster.get_score(importance_type='gain'))

def get_equal_share(pipes, h):
    """Return equal-share baseline % for this model at horizon h."""
    booster = pipes[h].named_steps['model'].model_.get_booster()
    n_features = len(booster.get_score(importance_type='gain'))
    return 100 / n_features

# ── Cross table ───────────────────────────────────────────────────────────
print(f"{'h':>4} {'Top OMNI feature':>20} {'OMNI gain%':>11} "
      f"{'neutron_counts (B)':>20} {'d_neutron (C)':>15} {'lag7 (D)':>10}")
print("─" * 85)

for h in K_HORIZONS:
    feat_name, omni_pct = get_top_omni(model_a_pipes, h)
    nc   = get_gain_pct(model_b_pipes, h, 'neutron_counts')
    dn   = get_gain_pct(model_c_pipes, h, 'd_neutron')
    lag7 = get_gain_pct(model_d_pipes, h, 'neutron_counts_lag7')
    eq_b = get_equal_share(model_b_pipes, h)
    eq_c = get_equal_share(model_c_pipes, h)
    eq_d = get_equal_share(model_d_pipes, h)
    print(f"{h:>4} {feat_name:>20} {omni_pct:>10.2f}% {nc:>20} {dn:>15} {lag7:>10}")

print(f"\n  Equal-share baseline: MODEL_B/C ~{eq_b:.2f}%, MODEL_D ~{eq_d:.2f}%")

   h     Top OMNI feature  OMNI gain%   neutron_counts (B)   d_neutron (C)   lag7 (D)
─────────────────────────────────────────────────────────────────────────────────────
   1           bz_acc_12h      30.91%                 1.13            0.32        0.0
   3           bz_acc_12h      27.89%                 1.26            1.03        0.0
   7           bz_acc_12h      23.38%                 1.87            1.28        0.0
  12           bz_acc_12h      16.50%                 3.61            1.14        0.0
  21           bz_acc_12h      11.89%                 6.89            1.87        0.0

  Equal-share baseline: MODEL_B/C ~4.76%, MODEL_D ~4.35%


The table below shows the feature selection status for all neutron-derived features — whether each passed the strict voting criterion (`selected_strict`, both methods agree) or the relaxed threshold (`selected_final`, at least one method), and which predictor configuration it is used in.

In [16]:
# ── Feature Selection consistency check ───────────────────────────────────
print("Feature Selection status — neutron features:")
print(f"{'Feature':>25} {'strict':>8} {'final':>8} {'Used in':>15}")
print("─" * 60)

neutron_features = {
    'neutron_counts'       : 'MODEL_B',
    'd_neutron'            : 'MODEL_C, MODEL_D',
    'neutron_counts_lag1'  : '—',
    'neutron_counts_lag3'  : 'MODEL_D',
    'neutron_counts_lag7'  : 'MODEL_D',
    'neutron_counts_lag12' : '—',
    'neutron_counts_lag21' : '—',
}

for feature, used_in in neutron_features.items():
    strict = feature in fs['selected_strict']
    final  = feature in fs['selected_final']
    print(f"  {feature:>25}: {str(strict):>8} {str(final):>8} {used_in:>15}")

Feature Selection status — neutron features:
                  Feature   strict    final         Used in
────────────────────────────────────────────────────────────
             neutron_counts:     True     True         MODEL_B
                  d_neutron:    False     True MODEL_C, MODEL_D
        neutron_counts_lag1:    False    False               —
        neutron_counts_lag3:     True     True         MODEL_D
        neutron_counts_lag7:    False    False         MODEL_D
       neutron_counts_lag12:     True     True               —
       neutron_counts_lag21:    False    False               —


The two tables above establish that neutron features are used by the model, but not whether the resulting difference from MODEL_A is larger than sampling noise. The Diebold-Mariano test below answers that directly, comparing MODEL_D against MODEL_A across all five horizons on both segments — the entire evaluation window in each case, not a subset.

In [17]:
def dm_test(pipes_a, pipes_b, features_a, features_b, mask, h):
    """One-sided DM test: positive stat = pipes_b better than pipes_a."""
    y_true = feat_data.loc[mask, f'dst_target_{h}h'].dropna()
    idx    = y_true.index
    pred_a = pipes_a[h].predict(feat_data.loc[idx, features_a])
    pred_b = pipes_b[h].predict(feat_data.loc[idx, features_b])
    stat, _ = diebold_mariano(y_true.values, pred_a, pred_b, h=h)
    return stat, 1 - stats.norm.cdf(stat)

def run_dm(pipes_b, features_b):
    print(f"{'h':>4} {'val_main DM':>12} {'val_main p':>12} {'sig':>5} "
          f"{'val_storm DM':>14} {'val_storm p':>12} {'sig':>5}")
    print("─" * 70)
    for h in K_HORIZONS:
        res = {seg: dm_test(model_a_pipes, pipes_b, OMNI_FEATURES, features_b, mask, h)
               for seg, mask in EVAL_SEGMENTS.items()}
        (dm_m, p_m), (dm_s, p_s) = res['val_main'], res['val_storm']
        print(f"{h:>4} {dm_m:>12.3f} {p_m:>12.4f} {'+'if p_m<0.05 else '—':>5} "
              f"{dm_s:>14.3f} {p_s:>12.4f} {'+'if p_s<0.05 else '—':>5}")

run_dm(model_d_pipes, MODEL_D_OMNI_HISTORY)

   h  val_main DM   val_main p   sig   val_storm DM  val_storm p   sig
──────────────────────────────────────────────────────────────────────
   1      -19.954       1.0000     —         -0.849       0.8021     —
   3      -18.794       1.0000     —          1.156       0.1238     —
   7       27.092       0.0000     +          2.910       0.0018     +
  12       17.421       0.0000     +         -0.246       0.5971     —
  21       45.701       0.0000     +         -1.335       0.9091     —


The Diebold-Mariano test above (`run_dm`) is computed over the **entire** validation segment, dominated numerically by quiet hours. To test the research question directly — does the neutron-augmented model improve storm-period predictions — the test below restricts the comparison to storm hours only ($D_{st}(t+h) < \text{STORM\_THR}$), for MODEL_B, MODEL_C and MODEL_D against MODEL_A, at $h=7$h.

In [18]:
def dm_test_storm_only(pipes_a, pipes_b, features_a, features_b, mask, h, storm_thr=STORM_THR):
    """
    DM тест, ограничен само до буреви часове (Dst(t+h) < storm_thr).
    Положителна статистика = pipes_b по-точен от pipes_a при амплитудата на бурята.
    """
    y_true = feat_data.loc[mask, f'dst_target_{h}h'].dropna()
    storm_mask = y_true < storm_thr
    idx = y_true[storm_mask].index

    n_storm = len(idx)
    if n_storm < 10:
        return None, None, n_storm

    pred_a = pipes_a[h].predict(feat_data.loc[idx, features_a])
    pred_b = pipes_b[h].predict(feat_data.loc[idx, features_b])
    stat, _ = diebold_mariano(y_true.loc[idx].values, pred_a, pred_b, h=h)
    p = 1 - stats.norm.cdf(stat)
    return stat, p, n_storm


def run_dm_storm_only(label, pipes_b, features_b, h=7):
    print(f"\n{label}:")
    print(f"{'Segment':<10} {'DM stat':>10} {'p':>10} {'n_storm':>9}")
    print("─" * 42)
    for seg_name, seg_mask in EVAL_SEGMENTS.items():
        stat, p, n = dm_test_storm_only(model_a_pipes, pipes_b, OMNI_FEATURES, features_b, seg_mask, h)
        if stat is None:
            print(f"{seg_name:<10} {'—':>10} {'—':>10} {n:>9}  (n<10, тестът е пропуснат)")
        else:
            sig = '+' if p < 0.05 else '—'
            print(f"{seg_name:<10} {stat:>10.3f} {p:>10.4f} {n:>9}   {sig}")


print(f"── DM тест (само буреви часове, Dst < {STORM_THR} nT), h=7h ──")
run_dm_storm_only('MODEL_B vs MODEL_A', model_b_pipes, MODEL_B_OMNI_RAW)
run_dm_storm_only('MODEL_C vs MODEL_A', model_c_pipes, MODEL_C_OMNI_DNEUTRON)
run_dm_storm_only('MODEL_D vs MODEL_A', model_d_pipes, MODEL_D_OMNI_HISTORY)

── DM тест (само буреви часове, Dst < -50.0 nT), h=7h ──

MODEL_B vs MODEL_A:
Segment       DM stat          p   n_storm
──────────────────────────────────────────
val_main       -3.309     0.9995      1165   —
val_storm       5.147     0.0000       190   +

MODEL_C vs MODEL_A:
Segment       DM stat          p   n_storm
──────────────────────────────────────────
val_main       -1.032     0.8491      1165   —
val_storm       4.510     0.0000       190   +

MODEL_D vs MODEL_A:
Segment       DM stat          p   n_storm
──────────────────────────────────────────
val_main       -1.671     0.9526      1165   —
val_storm       4.867     0.0000       190   +


`compute_delta_r2()` (defined earlier) compares only MODEL_A and MODEL_C. The analogous comparison for MODEL_D is added below, since the Ablation Study above evaluates only RMSE, not explained variance.

**Note:** this cell has not yet been executed — the resulting $R^2$ values are not yet available and should not be assumed before running it.

In [19]:
def compute_delta_r2_d(seg_name, h):
    r2_a = model_a_metrics[seg_name][h]['r2']
    r2_d = model_d_metrics[seg_name][h]['r2']
    return r2_a, r2_d, r2_d - r2_a

for seg_name in EVAL_SEGMENTS:
    print(f"\nΔR² = R²(MODEL_D) - R²(MODEL_A)  [{seg_name}]\n" + "="*45)
    print(f"{'h':>4} | {'R²(A)':>8} | {'R²(D)':>8} | {'ΔR²':>8}")
    print("-"*45)
    for h in K_HORIZONS:
        r2_a, r2_d, delta = compute_delta_r2_d(seg_name, h)
        print(f"{h:>3}h | {r2_a:>8.4f} | {r2_d:>8.4f} | {delta:>+8.4f}")


ΔR² = R²(MODEL_D) - R²(MODEL_A)  [val_main]
   h |    R²(A) |    R²(D) |      ΔR²
---------------------------------------------
  1h |   0.5092 |   0.4838 |  -0.0254
  3h |   0.4508 |   0.4136 |  -0.0372
  7h |   0.1544 |   0.2468 |  +0.0924
 12h |  -0.1900 |  -0.0909 |  +0.0991
 21h |  -0.5141 |  -0.2097 |  +0.3044

ΔR² = R²(MODEL_D) - R²(MODEL_A)  [val_storm]
   h |    R²(A) |    R²(D) |      ΔR²
---------------------------------------------
  1h |   0.6226 |   0.6174 |  -0.0052
  3h |   0.5519 |   0.5631 |  +0.0112
  7h |   0.3432 |   0.3710 |  +0.0278
 12h |   0.1958 |   0.1936 |  -0.0022
 21h |  -0.0226 |  -0.0475 |  -0.0250


`compute_metrics()` also returns `peak_timing_err` — the signed offset (in hours) between the predicted and observed timing of the $D_{st}$ minimum within the evaluation window. This metric was computed but not previously displayed for any model; it is checked here for MODEL_A, MODEL_C and MODEL_D at $h=7$h, Val_Storm (the single Halloween 2003 storm window).

In [20]:
for model_name, metrics in [('A', model_a_metrics), ('C', model_c_metrics), ('D', model_d_metrics)]:
    print(model_name, metrics['val_storm'][7]['peak_timing_err'])

A 1.0
C 1.0
D 1.0


> **Observations — Feature Importance Summary and Model/Horizon Selection.**
>
> Three independent lines of evidence now converge on the same horizon. `Val_Storm`'s $\Delta R^2$ peaked broadly across horizons but stayed positive throughout; `Val_Main`'s $\Delta R^2$ was positive only at $h=1,7,12$h, largest at $h=7$h; the Ablation Study showed all three neutron representations agreeing most tightly at $h=7$h. The whole-segment Diebold-Mariano test (Cell 51) now closes the loop: at $h=7$h, MODEL_D is significantly better than MODEL_A on **both** `Val_Main` (DM=+27.09, $p<0.0001$) and `Val_Storm` (DM=+2.91, $p=0.0018$) — the only horizon among the five where this holds simultaneously. At $h=1$h and $h=3$h, MODEL_D is significantly *worse* on `Val_Main` (DM $\approx -19$ to $-20$, $p=1.0000$ in the tested direction); at $h=12$h and $h=21$h it is significantly better on `Val_Main` but not on `Val_Storm`. $h=7$h is therefore not merely the horizon with the largest point estimate — it is the only one supported by significance on both evaluation segments at once.
>
> Restricting the same comparison to storm hours only (Cell 53, $Dst(t{+}7h) < -50$ nT) sharpens this further: all three neutron representations — MODEL_B, MODEL_C, MODEL_D — show significant improvement over MODEL_A on `Val_Storm`'s 190 storm hours ($p=0.0000$ for all three), and none show significant *harm* on `Val_Main`'s 1,165 storm hours (all $p \geq 0.85$ in the tested direction — well short of significance, unlike an earlier, since-superseded run of this analysis, where MODEL_D appeared to significantly underperform there). The storm-specific benefit is not an artefact of the single Halloween event driving an otherwise-noisy whole-segment average; it survives when the comparison is restricted to storm hours specifically, on both segments.
>
> $R^2$(MODEL_D) vs $R^2$(MODEL_A) (Cell 55) adds a horizon-dependent nuance: MODEL_D trails MODEL_A at $h=1,3$h on `Val_Main` ($\Delta R^2=-0.025, -0.037$) but pulls ahead sharply from $h=7$h onward, reaching $+0.304$ at $h=21$h — a materially larger gap than MODEL_C showed at the same horizon ($\Delta R^2=-0.019$, Cell 37). This suggests the lagged neutron history in MODEL_D specifically helps explained variance at longer horizons, even though MODEL_A's own $R^2$ is negative there ($-0.514$ at $h=21$h) — a large relative gain against a baseline that is itself barely usable at that lead time.
>
> Gain-based feature importance (Cells 46, 48) keeps all of this in proportion: neutron features stay below the equal-share baseline (~4.76% for MODEL_B/C, ~4.35% for MODEL_D) at every horizon up to $h=12$h, only exceeding it at $h=21$h — the horizon where MODEL_A's own predictive skill has already collapsed. `bz_acc_12h` dominates throughout (30.9% of total gain at $h=1$h, still 11.9% at $h=21$h). The peak-timing check (Cell 57) initially suggested MODEL_D localised the Halloween 2003 minimum more precisely than MODEL_A or MODEL_C (0 vs 1 hour offset); on a repeated, independently confirmed run this difference disappeared entirely (all three models: 1.0 hour) — a single-event metric (n=1) is not a stable basis for comparison, and is not treated as evidence here beyond this note.
>
> **Selection: MODEL_D at $h^*=7$h.** The lagged neutron history adds a small but significance-tested improvement over the differential flux alone (MODEL_C), specifically visible in the $R^2$ comparison from $h=7$h onward, while MODEL_C and MODEL_D perform near-identically on Storm RMSE at $h=7$h itself (+3.55 vs +3.86 nT). $h=7$h is carried forward to the Operational Forecasting Model stage.

## Summary — Methodological Verification

This section documents a reproducibility investigation carried out during this notebook's development, kept separate from the scientific narrative above so that the main argument reads without interruption. It is reported here, even though only partially resolved in its root cause, because it directly bears on how much confidence the numbers above deserve.

**The problem.** An early version of this notebook, run twice in immediate succession with identical code, identical `random_state=42`, and `n_jobs=-1` (the `XGBoostDst` default), produced different results between the two runs — differences large enough to flip the sign of $\Delta R^2$ at $h=12$h on `Val_Main` (from $-0.047$ to $+0.107$ across two runs of the same code). This is a direct violation of the reproducibility requirement stated in the project guidelines.

**What was ruled out.** Two controlled tests were run outside this notebook, fitting `XGBoostDst` twice in immediate succession on the actual project data (`feat_split.parquet`), first with XGBoost 3.3.0, then with 3.2.0 (matching this project's installed version exactly) — in both cases, and at both `n_jobs=1` and `n_jobs=-1`, the two repeated fits produced bit-identical predictions. This rules out XGBoost's own multi-threaded histogram construction as the source of the instability in this environment — the originally-suspected mechanism (non-associative floating-point summation across threads) does not reproduce here in isolation.

**What was confirmed stable.** Library versions (`xgboost 3.2.0`, `sklearn 1.9.0`, `pandas 2.3.3`) and MD5 checksums of both input artifacts (`feat_split.parquet`, `feature_selection_results.pkl`) were verified identical across two separate sessions in which the instability had previously appeared, ruling out silent input-data drift (e.g. an accidental re-run of `feature_selection.ipynb` between sessions) as the cause.

**What was actually verified end-to-end.** With `n_jobs=1` fixed throughout (`N_JOBS_VALIDATION` in Cell 0), this notebook was executed independently multiple times: Cells 12–40 matched exactly across three separate executions; Cells 41–59 matched exactly across two separate executions (the third attempt at completing this range was manually interrupted before finishing, for operational reasons — see below — and is not counted as a mismatch). Every number reported in the Observations above is drawn from this doubly-to-triply confirmed, matching set of runs.

**What remains open.** The root mechanism behind the original `n_jobs=-1` instability was not identified with certainty — it was ruled out at the XGBoost-fitting level specifically, but not traced to a specific alternative source (e.g. `LassoCV`'s own `n_jobs=-1` inside `feature_selection.ipynb` was flagged as an analogous, untested risk, but the confirmed-stable MD5 hashes of `feature_selection_results.pkl` across sessions make this an unlikely explanation for the specific instability observed here). Separately, `n_jobs=1` fitting is noticeably slower than `n_jobs=-1` and was observed to be interrupted (manually, due to wait time, not due to any raised exception) twice during development, always during a fresh multi-horizon XGBoost fit (MODEL_B, then MODEL_D) rather than during any of the lighter-weight cells — consistent with slow single-threaded fitting rather than a memory or correctness fault, but not independently timed or confirmed as the sole explanation.

**Practical conclusion for this project.** `n_jobs=1` is kept for every `XGBoostDst` instantiation in this notebook. The reproducibility requirement is satisfied for the results reported above, verified by repeated independent execution rather than by argument alone — but the underlying mechanism of the original instability, and therefore its applicability to other notebooks in this project using `n_jobs=-1` (e.g. `reference_performance_and_hpo.ipynb`), has not been established and should be checked before those results are relied upon at the same level of precision as the numbers in this notebook.

The specific library versions and input-artifact checksums referenced above, as recorded at the time of the confirmed runs:

In [21]:

print("xgboost:", xgboost.__version__)
print("sklearn:", sklearn.__version__)
print("pandas:", pd.__version__)

h1 = hashlib.md5(open('data/processed/feat_split.parquet', 'rb').read()).hexdigest()
h2 = hashlib.md5(open('models/feature_selection_results.pkl', 'rb').read()).hexdigest()

print("feat_split.parquet MD5:", h1)
print("feature_selection_results.pkl MD5:", h2)

xgboost: 3.2.0
sklearn: 1.9.0
pandas: 2.3.3
feat_split.parquet MD5: e2a1c9a2d13eda0594bfd686c8fcbbca
feature_selection_results.pkl MD5: 9d50dfc9381227bc2dcdb1fafa1f7242
